Implement model stacking using multiple classifiers without using built-in stacking functions, manually designing meta-models, optimizing base learners, and handling data leakage issues.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

print("Dataset shape:", df.shape)
print("Target values:", np.unique(y))

Dataset shape: (79, 6)
Target values: [0 1]


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 63
Testing samples: 16


In [3]:
models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000),
        {"C": [0.1, 1, 10]}
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"max_depth": [3, 5, None], "min_samples_leaf": [1, 5]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {"n_estimators": [100], "max_depth": [5, 10, None]}
    )
}

best_models = {}

for name, (model, params) in models.items():
    search = GridSearchCV(
        model, params, cv=3, scoring="accuracy", n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_

    print(name, "Best parameters:", search.best_params_)

Logistic Regression Best parameters: {'C': 1}
Decision Tree Best parameters: {'max_depth': 3, 'min_samples_leaf': 5}
Random Forest Best parameters: {'max_depth': 5, 'n_estimators': 100}


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

meta_train = np.zeros((len(X_train), len(best_models)))
meta_test = np.zeros((len(X_test), len(best_models)))

for j, (name, model) in enumerate(best_models.items()):
    test_fold_predictions = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        model.fit(X_train[train_idx], y_train[train_idx])

        meta_train[val_idx, j] = model.predict_proba(
            X_train[val_idx]
        )[:, 1]

        test_fold_predictions.append(
            model.predict_proba(X_test)[:, 1]
        )

    meta_test[:, j] = np.mean(test_fold_predictions, axis=0)

    print(name, "OOF predictions completed")

Logistic Regression OOF predictions completed
Decision Tree OOF predictions completed
Random Forest OOF predictions completed


In [5]:
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(meta_train, y_train)

print("Meta-model trained successfully")
print("Meta-model coefficients:", meta_model.coef_)

Meta-model trained successfully
Meta-model coefficients: [[1.21556082 0.71914948 1.63899992]]


In [6]:
y_pred = meta_model.predict(meta_test)

print("Stacking Model Results")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 4))

Stacking Model Results
Accuracy: 0.6875
Precision: 0.7333
Recall: 0.9167
F1 Score: 0.8148


In [7]:
for name, model in best_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    print("\n", name)
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))

print("\nStacking Model")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))


 Logistic Regression
Accuracy: 0.75

 Decision Tree
Accuracy: 0.6875

 Random Forest
Accuracy: 0.75

Stacking Model
Accuracy: 0.6875
